# 01 — Signal Processing & Bounce Detection

This notebook implements the **signal processing pipeline** described in the paper:
> **Sound-Based Spin Estimation in Table Tennis: Dataset and Real-Time Classification Pipeline** [1]
>
> Thomas Gossard, Julian Schmalzl, Andreas Ziegler, Andreas Zell

### Outline

1. **Data Overview** — dataset composition and surface/spin distributions
2. **Exploring Raw Audio** — loading and visualizing full match recordings
3. **High-Pass Filtering** — isolating bounce-characteristic frequencies (>10 kHz)
4. **Energy-Based Bounce Detection** — exponential moving average peak detection
5. **Comparing Detection Algorithms** — alternative approaches (thresholding, envelope, spectral)
6. **Individual Bounce Analysis** — extracting and inspecting 15 ms clips
7. **Spin Analysis** — frequency and spectrogram differences across spin types
8. **t-SNE Visualization** — embedding spectrograms in 2D space
9. **Conclusion** — summary of findings

In [1]:
import sys
sys.path.insert(0, '..')


import audio_utils
import plots


import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

from detectors import DecayAverageDetect,evaluate_detector

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.style.use("dark_background")
DATA_DIR = '../data'
SOUNDS_DIR = f'{DATA_DIR}/sounds'
RAW_DIR = f'{DATA_DIR}/raw_sounds'
SR = 44100

ImportError: cannot import name 'HFCDetect' from 'detectors.new_detectors' (/home/cl3t4p/Desktop/tt_detector/src/detectors/new_detectors.py)

## 1. Data Overview

The dataset contains **5702 annotated bounces** across 10 racket configurations, plus table, floor, and other surface types. Each sample is labeled with the surface type and spin direction (back, top, or none).

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/full.csv")

print(f"Total Samples : {len(df)}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Surface distribution
surface_counts = df['surface'].value_counts()
surface_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Distribution of Bounce Surfaces')
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Surface')
axes[0].tick_params(axis='x', rotation=45)

# Spin distribution (only where spin is annotated)
spin_counts = df['spin-direction'].value_counts()
spin_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Distribution of Spin Types')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Spin Direction')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 2. Exploring Raw Audio Recordings

Raw files contain entire match recordings. We load one to visualize the full waveform and understand the signal structure before applying any processing.

In [ ]:
# Load a raw recording with table bounces (STE-011 contains table bounces)
def get_random_raw():
    import random
    from pathlib import Path
    files = list(Path(RAW_DIR).glob("*.wav"))
    if not files:
        return None

    return random.choice(files)
#raw_file = get_random_raw()

raw_file_name = 'STE-011.wav'
raw_file = os.path.join(RAW_DIR, raw_file_name)
sig, sr = audio_utils.open_audio(raw_file)
waveform = sig.squeeze().numpy()

print(f"File name: {raw_file}")
print(f"Sample rate: {sr} Hz")
print(f"Sample length: {len(waveform)}")
print(f"Duration: {len(waveform)/sr:.2f} s")
print(f"Peak amplitude: {np.max(np.abs(waveform)):.4f}")

In [ ]:
# Full waveform view
fig, ax = plt.subplots(figsize=(14, 4))
plots.plot_waveform_ms(waveform, sr, title=f"Raw Recording: STE-011.wav ({len(waveform)/sr:.1f}s)", ax=ax)
plt.tight_layout()
plt.show()

# Listen to a short segment
display(ipd.Audio(waveform, rate=sr))

## 3. High-Pass Filtering

Table tennis ball bounces exhibit energy predominantly around **11 kHz and above**. The paper [1] applies a **5th-order Butterworth high-pass filter** at **10 kHz** to:
- Suppress low-frequency ambient noise (speech, room acoustics)
- Isolate the characteristic high-frequency bounce signature
- Improve the signal-to-noise ratio for the energy-based detector

This is critical because the detector relies on energy peaks — without filtering, speech or other low-frequency sounds would trigger false positives.

In [ ]:
# Apply high-pass filter
waveform_hp = audio_utils.highpass_filter(waveform, sr, cutoff=10_000, order=5)

# Compare a filtered and an unfiltered waveform
t_start, t_end = 0, 8  # seconds
s_start, s_end = int(t_start * sr), int(t_end * sr)
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

time = np.arange(s_start, s_end) / sr

# Original
axes[0].plot(time, waveform[s_start:s_end], linewidth=0.5)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Original Waveform (unfiltered)')
axes[0].grid(True, alpha=0.3)
# Filtered
axes[1].plot(time, waveform_hp[s_start:s_end], linewidth=0.5, color='orange')
axes[1].set_ylim(axes[0].get_ylim())   # same y-axis as original
axes[1].set_ylabel('Amplitude')
axes[1].set_title('High-Pass Filtered (>10 kHz, Butterworth 5th order)')
axes[1].grid(True, alpha=0.3)

# Overlay
axes[2].plot(time, waveform[s_start:s_end], linewidth=0.5, alpha=0.5, label='Original')
axes[2].plot(time, waveform_hp[s_start:s_end], linewidth=0.5, alpha=0.8, label='Filtered (>10 kHz)')
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('Comparison: Original vs. High-Pass Filtered')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

## 4. Energy-Based Bounce Detection

The core signal processing step: detecting ball bounces from continuous audio using **energy peak detection**.

#### Algorithm (Exponential Decay Average)
1. Divide the (filtered) signal into **1 ms** non-overlapping frames
2. Compute the **mean absolute energy** per frame: $e_k=\frac{1}{N}\sum |x_i|$
3. Maintain an **exponential moving average (EMA)** of energy:
   $$\bar{E}_{k+1} = \gamma \cdot \bar{E}_k + (1 - \gamma) \cdot e_k$$
   where $\gamma = 0.9$ is the decay factor
4. A **bounce** is detected when $e_k > \theta \cdot \bar{E}_k$ (threshold multiplier $\theta = 3$)
5. A **100 ms timeout** prevents double-triggering on the same event

The high-pass filter applied beforehand ensures the energy peaks correspond to ball impacts rather than noise.

In [ ]:
from src.detectors.energy_calculator import SimpleEnergyCalculator

simple_energy_calculator = SimpleEnergyCalculator()

# ---------- No high-pass ----------
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

time_full = np.arange(len(waveform)) / sr
no_hp_frame_energy = simple_energy_calculator.compute_frame_energy(
    waveform, sr, frame_ms=1.0
)
frame_time_no_hp = np.arange(len(no_hp_frame_energy)) * 1.0 / 1000.0

axes[0].plot(time_full, waveform, linewidth=0.3, alpha=0.7)
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Waveform (No High-Pass Filter)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(frame_time_no_hp, no_hp_frame_energy, linewidth=0.5, color="red")
axes[1].set_ylabel("Mean |Amplitude|")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Per-Frame Energy (1 ms frames)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ---------- High-pass ----------
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

time_full_hp = np.arange(len(waveform_hp)) / sr
hp_frame_energy = simple_energy_calculator.compute_frame_energy(
    waveform_hp, sr, frame_ms=1.0
)
frame_time_hp = np.arange(len(hp_frame_energy)) * 1.0 / 1000.0

axes[0].plot(time_full_hp, waveform_hp, linewidth=0.3, alpha=0.7)
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Waveform (High-Pass Filtered)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(frame_time_hp, hp_frame_energy, linewidth=0.5, color="red")
axes[1].set_ylabel("Mean |Amplitude|")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Per-Frame Energy (1 ms frames)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Run the Decay Average Detector
detector = DecayAverageDetect(
    decay=0.8,
    threshold_multiplier=3.0,
    timeout_ms=100.0,
    apply_highpass=True,
    highpass_cutoff=10_000,
    return_indexes=False,
)


peaks = detector.detect(waveform, sr)
print(f"Detected {len(peaks)} bounce events")
print(f"\nTimestamps (s): {[f'{p:.3f}' for p in peaks[:20]]}")

# Compare with ground truth
df_file = df[df['original-file'] == raw_file_name]
gt_timestamps = sorted(df_file['timestamp'].tolist())
print(f"\nGround truth bounces: {len(gt_timestamps)}")
print(f"GT timestamps (s): {[f'{t:.3f}' for t in gt_timestamps[:20]]}")

In [ ]:
# Detailed visualization of the detection process
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Waveform with detections and ground truth
time_arr = np.arange(len(waveform)) / sr
axes[0].plot(time_arr, waveform, linewidth=0.3, alpha=0.5, color='gray', label='Raw waveform')
for i, p in enumerate(peaks):
    axes[0].axvline(p, color='red', alpha=0.6, linewidth=1,
                    label='Detected' if i == 0 else None)
for i, gt in enumerate(gt_timestamps):
    axes[0].axvline(gt, color='green', alpha=0.4, linewidth=1, linestyle='--',
                    label='Ground Truth' if i == 0 else None)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Bounce Detection: Detected (red) vs Ground Truth (green)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Energy with EMA and threshold
energy = detector.energy_history_
avg = detector.avg_history_
thresh = detector.threshold_history_
t_e = np.arange(len(energy)) * 1.0 / 1000.0

axes[1].plot(t_e, energy, linewidth=0.5, color='black', label='Frame Energy')
axes[1].plot(t_e, avg, linewidth=1.2, color='orange', label='Energy Average (EMA)')
axes[1].plot(t_e, thresh, linewidth=1.0, color='red', linestyle='--', label='Detection Threshold')
for i, p in enumerate(peaks):
    idx = min(int(p * 1000), len(energy)-1)
    axes[1].plot(p, energy[idx], 'y*', markersize=12,
                label='Detected peak' if i == 0 else None)
axes[1].set_ylabel('Energy')
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Energy Peak Detection with Exponential Moving Average')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Zoomed-in view of a single bounce detection event
# Focus on the first detected bounce
if len(peaks) > 0:
    zoom_center = peaks[0]
    zoom_range = (zoom_center - 0.05, zoom_center + 0.05)  # 100ms window

    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    # Zoomed waveform
    mask_w = (time_arr >= zoom_range[0]) & (time_arr <= zoom_range[1])
    axes[0].plot(time_arr[mask_w], waveform[mask_w], linewidth=0.8, color='blue')
    axes[0].axvline(zoom_center, color='red', linewidth=2, alpha=0.7, label=f'Detection: {zoom_center:.4f}s')
    # Find closest GT
    closest_gt = min(gt_timestamps, key=lambda x: abs(x - zoom_center))
    axes[0].axvline(closest_gt, color='green', linewidth=2, linestyle='--', alpha=0.7,
                    label=f'Ground truth: {closest_gt:.4f}s')
    axes[0].set_ylabel('Amplitude')
    axes[0].set_title(f'Zoomed Bounce Detection (onset error: {abs(zoom_center-closest_gt)*1000:.2f} ms)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Zoomed energy
    mask_e = (t_e >= zoom_range[0]) & (t_e <= zoom_range[1])
    axes[1].plot(t_e[mask_e], energy[mask_e], linewidth=1.0, color='blue', label='Frame Energy')
    axes[1].plot(t_e[mask_e], avg[mask_e], linewidth=1.5, color='orange', label='EMA')
    axes[1].plot(t_e[mask_e], thresh[mask_e], linewidth=1.2, color='red', linestyle='--', label='Threshold')
    axes[1].axvline(zoom_center, color='red', linewidth=2, alpha=0.5)
    axes[1].set_ylabel('Energy')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title('Energy & Threshold (Zoomed)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5. Comparing Detection Algorithms

We implemented four approaches and compare them:
- **Adjacent Frame** — simplest, compares consecutive frames
- **Moving Average** — sliding window of past 100 frames
- **Decay Average (EMA)** — exponential moving average — best adaptive behavior
- **Scipy Peaks** — `find_peaks` with prominence threshold

In [ ]:
# Compare all four detectors
from detectors.other_detectors import AdjacentFrameDetect,MovingAverageDetect,ScipyPeakDetect

detectors = {
    'Adjacent Frame': AdjacentFrameDetect(threshold=0.005, timeout_ms=20),
    'Moving Average': MovingAverageDetect(threshold_multiplier=5.0, timeout_ms=50),
    'Decay Average (EMA)': DecayAverageDetect(),
    'Scipy Peaks': ScipyPeakDetect(min_distance_ms=100, prominence=0.01,apply_highpass=False),
}

# Apply highpass to the raw waveform for non-EMA detectors
waveform_filtered = audio_utils.highpass_filter(waveform, sr, cutoff=10000.0)

results = {}
for name, det in detectors.items():
    if name != 'Decay Average (EMA)':
        pks = det.detect(waveform, sr)  # EMA applies its own HPF
    else:
        pks = det.detect(waveform_filtered, sr)
    results[name] = pks
    print(f"{name:25s}: {len(pks)} detections")

print(f"\nGround truth: {len(gt_timestamps)} bounces")

In [ ]:
# Visual comparison of detector outputs
fig, axes = plt.subplots(len(detectors) + 1, 1, figsize=(14, 3 * (len(detectors) + 1)),
                         sharex=True)

# Ground truth
axes[0].plot(time_arr, waveform, linewidth=0.2, color='gray')
for gt in gt_timestamps:
    axes[0].axvline(gt, color='green', alpha=0.5, linewidth=1)
axes[0].set_title(f'Ground Truth ({len(gt_timestamps)} bounces)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Each detector
colors = ['blue', 'orange', 'red', 'purple']
for i, (name, pks) in enumerate(results.items()):
    axes[i+1].plot(time_arr, waveform, linewidth=0.2, color='gray')
    for p in pks:
        axes[i+1].axvline(p, color=colors[i], alpha=0.6, linewidth=1)
    axes[i+1].set_title(f'{name} ({len(pks)} detections)')
    axes[i+1].set_ylabel('Amplitude')
    axes[i+1].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative evaluation of the decay average detector
eval_result = evaluate_detector(
    DecayAverageDetect(decay=0.9, threshold_multiplier=3.0, timeout_ms=100),
    raw_file,
    f'{DATA_DIR}/full.csv',
    sr=SR,
    tolerance_ms=5.0
)

print("Decay Average Detector Evaluation (tolerance=5ms):")
print(f"  Precision:           {eval_result['precision']:.3f}")
print(f"  Recall:              {eval_result['recall']:.3f}")
print(f"  F1 Score:            {eval_result['f1']:.3f}")
print(f"  Mean Onset Error:    {eval_result['mean_onset_error_ms']:.2f} ms")
print(f"  True Positives:      {eval_result['true_positives']}")
print(f"  False Positives:     {eval_result['false_positives']}")
print(f"  False Negatives:     {eval_result['false_negatives']}")

## 5b. New Detection Methods (Spectral & Statistical)

Beyond energy-threshold approaches, the literature offers **spectral** and **statistical** onset detectors.
We evaluate four additional algorithms:

| Detector | Method | Reference |
|---|---|---|
| **Spectral Flux** | Half-wave-rectified frame-to-frame STFT magnitude increase | Dixon (2006) "Onset Detection Revisited", DAFx-06 |
| **HFC** | High Frequency Content — spectral energy weighted by bin index, emphasising high-freq transients | Masri (1996); reviewed in Bello et al. (2005) IEEE TSAP |
| **SuperFlux** | Spectral flux with maximum-filter vibrato suppression on Mel bands | Böck & Widmer (2013) "Maximum Filter Vibrato Suppression", DAFx-13 |

> **Note:** Spectral methods detect the *onset* of energy change (~10-15 ms before the peak),
> so we evaluate at both **5 ms** (strict) and **20 ms** (relaxed) tolerance.

In [ ]:
from detectors.new_detectors import SpectralFluxDetect, SuperFluxDetect

new_detectors = {
    'Spectral Flux': SpectralFluxDetect(
        n_fft=512, hop_length=256, threshold_multiplier=5.0,
    ),
    'SuperFlux': SuperFluxDetect(
        n_fft=1024, n_bands=24, delta=1.0, threshold_multiplier=1.0,
    )
}

# Include the baseline for comparison
all_detectors = {
    'Decay Average (baseline)': DecayAverageDetect(),
    **new_detectors,
}

# Run detections on the current raw file
new_results = {}
for name, det in all_detectors.items():
    pks = det.detect(waveform, sr)
    new_results[name] = pks
    print(f"{name:30s}: {len(pks)} detections")

In [ ]:
# Visual comparison: ground truth vs all detectors
fig, axes = plt.subplots(len(all_detectors) + 1, 1,
                         figsize=(14, 3 * (len(all_detectors) + 1)),
                         sharex=True)

# Ground truth
axes[0].plot(time_arr, waveform, linewidth=0.2, color='gray')
for gt in gt_timestamps:
    axes[0].axvline(gt, color='green', alpha=0.5, linewidth=1)
axes[0].set_title(f'Ground Truth ({len(gt_timestamps)} bounces)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

# Each detector
colors = ['red', 'dodgerblue', 'orange', 'magenta', 'cyan']
for i, (name, pks) in enumerate(new_results.items()):
    axes[i+1].plot(time_arr, waveform, linewidth=0.2, color='gray')
    for p in pks:
        axes[i+1].axvline(p, color=colors[i], alpha=0.6, linewidth=1)
    axes[i+1].set_title(f'{name} ({len(pks)} detections)')
    axes[i+1].set_ylabel('Amplitude')
    axes[i+1].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative evaluation across all 10 files at both tolerances
raw_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.startswith('0') and f.split('.')[0].isdigit()
])

rows = []
for tol in [5.0, 20.0]:
    for name, det in all_detectors.items():
        file_evals = []
        for rf in raw_files:
            r = evaluate_detector(det, os.path.join(RAW_DIR, rf),
                                  f'{DATA_DIR}/full.csv', tolerance_ms=tol)
            file_evals.append(r)
        rows.append({
            'Tolerance (ms)': tol,
            'Detector': name,
            'Precision': np.mean([r['precision'] for r in file_evals]),
            'Recall':    np.mean([r['recall'] for r in file_evals]),
            'F1':        np.mean([r['f1'] for r in file_evals]),
            'Onset Err (ms)': np.nanmean([r['mean_onset_error_ms'] for r in file_evals]),
            'TP': sum(r['true_positives'] for r in file_evals),
            'FP': sum(r['false_positives'] for r in file_evals),
            'FN': sum(r['false_negatives'] for r in file_evals),
        })

eval_df = pd.DataFrame(rows)

for tol in [5.0, 20.0]:
    print(f"\n{'='*75}")
    print(f"  Tolerance = {tol} ms   |   Files: {', '.join(raw_files)}")
    print(f"{'='*75}")
    sub = eval_df[eval_df['Tolerance (ms)'] == tol].copy()
    sub = sub.drop(columns=['Tolerance (ms)'])
    print(sub.to_string(index=False, float_format='%.3f'))

## 5c. SuperFlux vs Decay Average — Large-Scale Comparison

The previous evaluation only covered the 10 basic numbered files.
Here we compare **SuperFlux** and **Decay Average** across **all annotated raw files**
(115 files, ~5700 bounces) to get a more robust picture of detection accuracy.

In [ ]:
# Gather all annotated raw files (files that appear in full.csv)
df_full = pd.read_csv(f"{DATA_DIR}/full.csv")
annotated_files = sorted([
    f for f in os.listdir(RAW_DIR)
    if f.startswith('0')
])

# Keep only files that actually exist on disk
annotated_files = [
    f for f in annotated_files
    if os.path.isfile(os.path.join(RAW_DIR, f))
]
print(f"Evaluating on {len(annotated_files)} annotated files "
      f"({df_full[df_full['original-file'].isin(annotated_files)]['original-file'].count()} total bounces)")

# Define the two detectors to compare
compare_detectors = {
    'Decay Average': DecayAverageDetect(),
    'SuperFlux': SuperFluxDetect(
        n_fft=1024, n_bands=24, delta=1.0, threshold_multiplier=1.0,
    ),
}

# Evaluate each detector on every file at both tolerances
comparison_rows = []
per_file_rows = []
precision = [7.0,20.0]

for tol in precision:
    for name, det in compare_detectors.items():
        file_evals = []
        for rf in annotated_files:
            r = evaluate_detector(det, os.path.join(RAW_DIR, rf),
                                  f'{DATA_DIR}/full.csv', tolerance_ms=tol)
            file_evals.append(r)
            per_file_rows.append({
                'Tolerance (ms)': tol,
                'Detector': name,
                'File': rf,
                **r,
            })

        comparison_rows.append({
            'Tolerance (ms)': tol,
            'Detector': name,
            'Precision': np.mean([r['precision'] for r in file_evals]),
            'Recall':    np.mean([r['recall'] for r in file_evals]),
            'F1':        np.mean([r['f1'] for r in file_evals]),
            'Onset Err (ms)': np.nanmean([r['mean_onset_error_ms'] for r in file_evals]),
            'TP': sum(r['true_positives'] for r in file_evals),
            'FP': sum(r['false_positives'] for r in file_evals),
            'FN': sum(r['false_negatives'] for r in file_evals),
        })

comp_df = pd.DataFrame(comparison_rows)
per_file_df = pd.DataFrame(per_file_rows)

for tol in precision:
    print(f"\n{'='*80}")
    print(f"  SuperFlux vs Decay Average  |  Tolerance = {tol} ms  |  {len(annotated_files)} files")
    print(f"{'='*80}")
    sub = comp_df[comp_df['Tolerance (ms)'] == tol].copy()
    sub = sub.drop(columns=['Tolerance (ms)'])
    print(sub.to_string(index=False, float_format='%.3f'))

In [ ]:
# Visual comparison: bar charts for Precision, Recall, F1 at both tolerances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, tol in zip(axes, precision):
    sub = comp_df[comp_df['Tolerance (ms)'] == tol]
    x = np.arange(3)
    width = 0.3
    metrics = ['Precision', 'Recall', 'F1']

    for i, (_, row) in enumerate(sub.iterrows()):
        vals = [row[m] for m in metrics]
        ax.bar(x + i * width, vals, width, label=row['Detector'])
        # Add value labels on bars
        for j, v in enumerate(vals):
            ax.text(x[j] + i * width, v + 0.01, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(metrics)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'Tolerance = {tol} ms')
    ax.legend()

fig.suptitle(f'SuperFlux vs Decay Average — {len(annotated_files)} files', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Per-file F1 comparison (tolerance = 20 ms)
tol_show = 20.0
pf = per_file_df[per_file_df['Tolerance (ms)'] == tol_show].copy()

da_f1 = pf[pf['Detector'] == 'Decay Average'].set_index('File')['f1']
sf_f1 = pf[pf['Detector'] == 'SuperFlux'].set_index('File')['f1']

# Sort by Decay Average F1 for readability
common_files = sorted(da_f1.index, key=lambda f: da_f1[f])

fig, ax = plt.subplots(figsize=(14, max(6, len(common_files) * 0.18)))
y = np.arange(len(common_files))

ax.barh(y - 0.15, [da_f1[f] for f in common_files], 0.3, label='Decay Average', alpha=0.85)
ax.barh(y + 0.15, [sf_f1[f] for f in common_files], 0.3, label='SuperFlux', alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(common_files, fontsize=6)
ax.set_xlabel('F1 Score')
ax.set_xlim(0, 1.05)
ax.set_title(f'Per-file F1 Score — Tolerance = {tol_show} ms')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

# Summary: how often each detector wins
da_wins = sum(1 for f in common_files if da_f1[f] > sf_f1[f])
sf_wins = sum(1 for f in common_files if sf_f1[f] > da_f1[f])
ties = len(common_files) - da_wins - sf_wins
print(f"\nPer-file wins (tol={tol_show} ms): "
      f"Decay Average={da_wins}, SuperFlux={sf_wins}, Ties={ties}")

## 6. Individual Bounce Analysis

Each bounce clip is **15 ms** long (661 samples at 44.1 kHz), extracted starting 3 ms before the annotated onset. We examine these clips through waveforms, Mel spectrograms, and MFCCs.

In [ ]:
# Load a few individual bounce sounds
bounce_ids = [2000, 2359, 3000]
fig, axes = plt.subplots(len(bounce_ids), 1, figsize=(12, 2.5 * len(bounce_ids)))

for i, bid in enumerate(bounce_ids):
    path = os.path.join(SOUNDS_DIR, f'{bid}.wav')
    if not os.path.exists(path):
        continue
    sig_b, sr_b = audio_utils.open_audio(path)
    w = sig_b.squeeze().numpy()

    row = df[df['bounce-id'] == bid].iloc[0]
    label = f"ID={bid}, surface={row['surface']}, spin={row['spin-direction']}"
    plots.plot_waveform_ms(w, sr_b, title=label, ax=axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Detailed view of a single bounce: waveform, mel spectrogram, MFCC
bounce_id = 2359
path = os.path.join(SOUNDS_DIR, f'{bounce_id}.wav')
aud = audio_utils.open_audio(path)
aud = audio_utils.pad_trunc(aud, 661)
waveform = aud[0].numpy()

row = df[df['bounce-id'] == bounce_id].iloc[0]

fig, axes = plt.subplots(3, 1, figsize=(10, 10))

# Waveform
plots.plot_waveform_ms(waveform, SR, title=f"Waveform", ax=axes[0])

# Mel spectrogram
mel = audio_utils.mel_spectro_gram(waveform)
plots.plot_mel_spectrogram(mel, SR, hop_length=64,
                     title=f"Mel Spectrogram (64 Mel bins, hop=64)", ax=axes[1])

# MFCC
mfcc_data = audio_utils.mfcc(aud)
plots.plot_mfcc(mfcc_data, SR, hop_length=64, title="MFCC (40 coefficients)", ax=axes[2])

plt.suptitle(f"Three Representations of Bounce Sound (ID={bounce_id}, {row['surface']} , spin-direction : {row['spin-direction']})",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Play the sound (upsampled for audibility)
display(ipd.Audio(waveform, rate=SR))

## 7. Spin Analysis

The paper [1] shows that **spin presence can be detected from sound** (backspin/topspin vs. no spin), though **spin direction** is harder to distinguish acoustically. Topspin and backspin both show stronger high-frequency components compared to no-spin bounces.

We compare frequency spectra and spectrograms across surface types and spin directions.

In [ ]:
# Compare frequency spectra across surface types
surfaces_to_compare = ['table', 'floor', 'racket', 'other']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, surface in enumerate(surfaces_to_compare):
    ax = axes[idx // 2][idx % 2]
    df_surf = df[df['surface'] == surface].head(100)  # Take up to 100 samples

    spectra = []
    for _, row in df_surf.iterrows():
        bid = int(row['bounce-id'])
        path = os.path.join(SOUNDS_DIR, f'{bid}.wav')
        if not os.path.exists(path):
            continue
        aud = audio_utils.open_audio(path)
        aud = audio_utils.pad_trunc(aud, 661)
        w = aud[0].squeeze().numpy()
        fft_mag = np.abs(np.fft.rfft(w)) / len(w)
        spectra.append(fft_mag)

    if spectra:
        spectra = np.array(spectra)
        freqs = np.fft.rfftfreq(661, d=1.0/SR)
        mean_spec = np.mean(spectra, axis=0)
        std_spec = np.std(spectra, axis=0)

        ax.plot(freqs, mean_spec, linewidth=1.0)
        ax.fill_between(freqs, mean_spec - std_spec, mean_spec + std_spec, alpha=0.3)
        ax.set_title(f'Average Spectrum: {surface} (n={len(spectra)})')
        ax.set_xlabel('Frequency (Hz)')
        ax.set_ylabel('Magnitude')
        ax.grid(True, alpha=0.3)

plt.suptitle('Frequency Spectra by Surface Type', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare spectrograms for different spin types
spin_types = ['back', 'none', 'top']
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row_idx, spin in enumerate(spin_types):
    df_spin = df[df['spin-direction'] == spin]
    if len(df_spin) == 0:
        continue
    samples = df_spin.sample(min(3, len(df_spin)), random_state=42)

    for col_idx, (_, row) in enumerate(samples.iterrows()):
        bid = int(row['bounce-id'])
        path = os.path.join(SOUNDS_DIR, f'{bid}.wav')
        if not os.path.exists(path):
            continue
        aud = audio_utils.open_audio(path)
        aud = audio_utils.pad_trunc(aud, 661)
        w = aud[0].squeeze().numpy()
        mel = audio_utils.mel_spectro_gram(w)
        plots.plot_mel_spectrogram(mel, SR, hop_length=64,
                            title=f"Spin: {spin} (ID={bid}, {row['surface']})",
                            ax=axes[row_idx][col_idx])

plt.suptitle('Mel Spectrograms by Spin Type (cf. Paper Fig. 4)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Average frequency spectrum comparison by spin type
fig, ax = plt.subplots(figsize=(12, 5))
colors_spin = {'back': 'blue', 'none': 'green', 'top': 'red'}

for spin_type in spin_types:
    df_spin = df[df['spin-direction'] == spin_type]
    if len(df_spin) == 0:
        continue
    df_spin = df_spin.sample(min(100, len(df_spin)), random_state=42)

    spectra = []
    for _, row in df_spin.iterrows():
        bid = int(row['bounce-id'])
        path = os.path.join(SOUNDS_DIR, f'{bid}.wav')
        if not os.path.exists(path):
            continue
        aud = audio_utils.open_audio(path)
        aud = audio_utils.pad_trunc(aud, 661)
        w = aud[0].squeeze().numpy()
        fft_mag = np.abs(np.fft.rfft(w)) / len(w)
        spectra.append(fft_mag)

    spectra = np.array(spectra)
    freqs = np.fft.rfftfreq(661, d=1.0/SR)
    mean_spec = np.mean(spectra, axis=0)
    std_spec = np.std(spectra, axis=0)

    ax.plot(freqs, mean_spec, linewidth=1.5, color=colors_spin[spin_type],
            label=f'{spin_type} (n={len(spectra)})')
    ax.fill_between(freqs, mean_spec - std_spec, mean_spec + std_spec,
                    alpha=0.15, color=colors_spin[spin_type])

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude')
ax.set_title('Average Frequency Spectrum by Spin Type')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 8. t-SNE Visualization of Bounce Embeddings

To verify that bounce sounds are distinguishable in feature space, we apply **t-SNE** to the normalized Mel spectrograms. The paper (Fig. 3) shows:
- Racket types form **distinct clusters** for simple vertical bounces
- Spin presence (yes/no) is separable, but direction (back vs. top) overlaps

In [ ]:
from sklearn.manifold import TSNE

# Collect Mel spectrograms for a subset of samples
n_samples = min(1500, len(df))
df_sample = df.sample(n_samples, random_state=42)

features = []
surface_labels = []
spin_labels = []
valid_ids = []

for _, row in df_sample.iterrows():
    bid = int(row['bounce-id'])
    path = os.path.join(SOUNDS_DIR, f'{bid}.wav')
    if not os.path.exists(path):
        continue
    aud = audio_utils.open_audio(path)
    aud = audio_utils.pad_trunc(aud, 661)
    w = aud[0].squeeze().numpy()
    mel = audio_utils.mel_spectro_gram(w)
    features.append(mel.squeeze().numpy().flatten())
    surface_labels.append(str(row['surface']))
    spin_labels.append(str(row['spin-direction']))
    valid_ids.append(bid)

features = np.array(features)
# Normalize
features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-8)

print(f"Collected {len(features)} samples for t-SNE")
print(f"Feature shape: {features.shape}")

In [ ]:
# Compute t-SNE
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter_without_progress=1000)
embeddings = tsne.fit_transform(features)

# --- Surface clustering ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# By surface type (simplified: table, floor, racket, other)
surface_simple = []
for s in surface_labels:
    if s == 'table':
        surface_simple.append('table')
    elif s == 'floor':
        surface_simple.append('floor')
    elif s == 'racket':
        surface_simple.append('racket')
    else:
        surface_simple.append('other')

surface_simple = np.array(surface_simple)
for label in np.unique(surface_simple):
    mask = surface_simple == label
    axes[0].scatter(embeddings[mask, 0], embeddings[mask, 1],
                   label=label, alpha=0.5, s=15)
axes[0].set_title('t-SNE: Surface Type Clustering')
axes[0].legend(markerscale=3)
axes[0].grid(True, alpha=0.3)

# By spin type
spin_arr = np.array(spin_labels)
spin_colors = {'back': 'blue', 'none': 'green', 'top': 'red'}
for label in ['back', 'none', 'top']:
    mask = spin_arr == label
    if mask.sum() > 0:
        axes[1].scatter(embeddings[mask, 0], embeddings[mask, 1],
                       label=label, alpha=0.5, s=15,
                       color=spin_colors.get(label, 'gray'))
axes[1].set_title('t-SNE: Spin Type Clustering')
axes[1].legend(markerscale=3)
axes[1].grid(True, alpha=0.3)

plt.suptitle('t-SNE Visualization of Mel Spectrogram Features (cf. Paper Fig. 3)', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Conclusion

This notebook walked through the full signal processing pipeline for table tennis bounce detection and analysis:

- **High-pass filtering at 10 kHz** effectively isolates bounce signatures from ambient noise, confirming that ball impacts concentrate energy above 11 kHz
- **Exponential decay average (EMA) detection** provides the best balance of adaptivity and robustness among the four original approaches tested, achieving F1 = 0.948 at 5 ms tolerance
- **Spectral onset detectors (Spectral Flux, HFC)** match or slightly exceed the baseline at 20 ms tolerance (F1 ≈ 0.953), but detect the onset ~10-15 ms before the annotated peak, penalising them at strict 5 ms tolerance
- **SuperFlux** underperforms on impulsive bounce signals — its vibrato suppression is designed for musical contexts and generates excessive false positives here
- **CUSUM** offers moderate performance; its statistical change-point approach is less effective against the variable background noise of match recordings
- **15 ms bounce clips** contain sufficient information for classification — Mel spectrograms and MFCCs capture discriminative features across surface types
- **Spin presence is acoustically distinguishable** from no-spin bounces (stronger high-frequency components), but **spin direction (back vs. top) remains difficult to separate** from frequency analysis alone
- **t-SNE embeddings** confirm that surface types form distinct clusters in Mel spectrogram feature space, supporting the feasibility of a classifier

### Next Steps

These findings motivate the classification pipeline explored in the next notebook, where we train models on Mel spectrogram features to classify surface type and spin direction from individual bounce sounds.